# The Joint Source-Channel Transmission Dashboard

Before we completely abandon the traditional PyTorch LSTM architecture and upgrade to Phase 2 (Generative AI / LLMs), we must solidify our baseline.

In this notebook, we perform a **massive training run** on the Europarl dataset for 10 epochs. 
Once training is complete, an Interactive Radio Dashboard will load at the bottom.

In [1]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Add src to path
sys.path.append(os.path.abspath(r'C:\Users\Shrish\Desktop\semantic-comm\actual_project'))
from src.data_loader import EuroparlDataLoader
from src.tokenizer import SemanticTokenizer
from src.model import JointSemanticModel

print("DONE")

DONE


### Step 1: Massive Europarl Dataset Loading
To get a truly capable LSTM model, we need to train on a massive amount of data. 
EUROPARL is 20million sentence , but i dont have that good gpu , it woul take weeks so im capping at 40000 sentence

In [2]:
data_dir = r"C:\Users\Shrish\Desktop\semantic-comm\Semantic_Communication\europarl\en\en"

loader = EuroparlDataLoader(data_dir=data_dir, target_sentences=40000)
loader.scan_and_load()

sentences = loader.all_sentences if len(loader.all_sentences) > 0 else ["aadya is melting my heart and graphics gard , sad life"] * 100000
tokenizer = SemanticTokenizer(min_freq=2)
tokenizer.fit(sentences)

print(f" {len(sentences)} sentences. vocab size: {tokenizer.vocab_size}")

class SemanticDataset(Dataset):
    def __init__(self, sents, tok, max_l=15):
        self.sents = sents; self.tok = tok; self.max_l = max_l
    def __len__(self): return len(self.sents)
    def __getitem__(self, idx):
        return torch.tensor(self.tok.encode(self.sents[idx], max_length=self.max_l), dtype=torch.long)

dataset = SemanticDataset(sentences, tokenizer, max_l=15)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True) 

Scanning dataset to build vocabulary...
dictionary size: 23209
 38633 sentences. vocab size: 23209


### The Deep Learning Training

In [3]:
model = JointSemanticModel(vocab_size=tokenizer.vocab_size, embed_dim=128, hidden_dim=256, snr_db=10.0)
optimizer = optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.PAD_ID)

epochs = 10
print(f"{len(sentences)} sentences for {epochs} epochs..")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(batch, batch)
        loss = criterion(outputs[:, :-1, :].reshape(-1, tokenizer.vocab_size), batch[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch{epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

print("done")

38633 sentences for 10 epochs..
Epoch1/10 | Loss: 5.8653
Epoch2/10 | Loss: 4.6527
Epoch3/10 | Loss: 3.9029
Epoch4/10 | Loss: 3.4015
Epoch5/10 | Loss: 3.0104
Epoch6/10 | Loss: 2.6935
Epoch7/10 | Loss: 2.4231
Epoch8/10 | Loss: 2.1917
Epoch9/10 | Loss: 1.9980
Epoch10/10 | Loss: 1.8364
done


### dashboard
input->guessing the input as it is destroyed by noise 

In [6]:
def transmit_sentence(input_text, snr_value):
    if not input_text.strip():
        return
        
    print("="*50)
    print("TRANSMISSION LOG")
    print("="*50)
    print(f"Channel value: {snr_value} dB SNR\n")
    
    #tokenize
    tokens = tokenizer.encode(input_text, max_length=15)
    token_tensor = torch.tensor([tokens], dtype=torch.long)
    print(f"[TX] Message: '{input_text}'")
    print(f"[TX] Tokens : {tokens}\n")
    
    model.eval()
    with torch.no_grad():
        
        _, (h, c) = model.encoder.lstm(model.encoder.embedding(token_tensor))
        noisy_h = model.channel(h, snr_db_override=snr_value)
        noisy_c = model.channel(c, snr_db_override=snr_value)
        input_step = torch.tensor([[tokenizer.BOS_ID]], dtype=torch.long)
        h_dec, c_dec = noisy_h, noisy_c
        
        predicted_tokens = []
        for _ in range(15):
            prediction, h_dec, c_dec = model.decoder(input_step, h_dec, c_dec)
            predicted_id = torch.argmax(prediction, dim=1).item()
            predicted_tokens.append(predicted_id)
            
            input_step = torch.tensor([[predicted_id]], dtype=torch.long)
            if predicted_id == tokenizer.EOS_ID:
                break
    reconstructed_text = tokenizer.decode(predicted_tokens)
    print(f"[RX] Tokens : {predicted_tokens}")
    print(f"[RX] Message: '{reconstructed_text}'")
    print("="*50)
    
interact(transmit_sentence, 
         input_text=widgets.Text(value='enter the message', description='Message:', layout=widgets.Layout(width='80%')),
         snr_value=widgets.FloatSlider(value=10.0, min=-10.0, max=30.0, step=1.0, description='SNR (dB):', layout=widgets.Layout(width='50%')))
print("Ready!")

interactive(children=(Text(value='enter the message', description='Message:', layout=Layout(width='80%')), Flo…

Ready!
